In [51]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import plotly.express as px

elo = pd.read_csv("../data/raw/clubelo/team_history/Man_United.csv")
display(elo)


pd.set_option("display.max_columns", None)

,from,rank,team,country,level,elo,to,season,league,manager,manager_nationality
0,1946-07-07,NaN,Man United,ENG,1,1551.140259,1946-08-31,NaN,NaN,Sir Matt Busby,Scotland
1,1946-09-01,NaN,Man United,ENG,1,1556.314697,1946-09-04,1946-47,ENG-Premier League,Sir Matt Busby,Scotland
2,1946-09-05,NaN,Man United,ENG,1,1573.838867,1946-09-07,1946-47,ENG-Premier League,Sir Matt Busby,Scotland
3,1946-09-08,NaN,Man United,ENG,1,1587.939209,1946-09-11,1946-47,ENG-Premier League,Sir Matt Busby,Scotland
4,1946-09-12,NaN,Man United,ENG,1,1598.377075,1946-09-14,1946-47,ENG-Premier League,Sir Matt Busby,Scotland
...,...,...,...,...,...,...,...,...,...,...,...
6224,2026-03-22,9.0,Man United,ENG,1,1880.537476,2026-03-22,2025-26,ENG-Premier League,Michael Carrick,England
6225,2026-03-23,10.0,Man United,ENG,1,1880.537476,2026-04-13,2025-26,ENG-Premier League,Michael Carrick,England
6226,2026-04-14,10.0,Man United,ENG,1,1880.537476,2026-04-18,2025-26,ENG-Premier League,Michael Carrick,England
6227,2026-04-19,10.0,Man United,ENG,1,1880.537476,2026-04-20,2025-26,ENG-Premier League,Michael Carrick,England


In [52]:
#Elo since Ferguson Retired
import plotly.graph_objects as go

post_fergie = elo.loc[elo["from"] > "2013-06-30"].copy()
post_fergie["from"] = pd.to_datetime(post_fergie["from"])
post_fergie["elo"] = pd.to_numeric(post_fergie["elo"], errors="coerce")
post_fergie["manager"] = post_fergie["manager"].fillna("Unknown").replace("", "Unknown")
post_fergie = post_fergie.dropna(subset=["from", "elo"]).sort_values("from").reset_index(drop=True)
post_fergie["manager_change"] = post_fergie["manager"].ne(post_fergie["manager"].shift())
post_fergie["manager_stint_id"] = post_fergie["manager_change"].cumsum()

stint_lookup = post_fergie[["manager", "manager_stint_id"]].drop_duplicates().copy()
stint_lookup["manager_stint_number"] = stint_lookup.groupby("manager").cumcount() + 1
stint_lookup["manager_stint_count"] = stint_lookup.groupby("manager")["manager_stint_id"].transform("size")
stint_lookup["manager_stint_label"] = np.where(
    stint_lookup["manager_stint_count"] > 1,
    stint_lookup["manager"] + " (Stint " + stint_lookup["manager_stint_number"].astype(str) + ")",
    stint_lookup["manager"],
)
post_fergie = post_fergie.merge(
    stint_lookup[["manager_stint_id", "manager_stint_label"]],
    on="manager_stint_id",
    how="left",
)

sns.set_theme(
    style="darkgrid",
    context="paper",
    palette="colorblind"
    )
sns.despine()

manager_order = post_fergie["manager"].drop_duplicates().tolist()
color_sequence = px.colors.qualitative.Safe + px.colors.qualitative.Vivid + px.colors.qualitative.Set3
manager_colors = {
    manager: color_sequence[idx % len(color_sequence)]
    for idx, manager in enumerate(manager_order)
}

def get_stint_extrema(df):
    extrema_rows = []
    for stint_label, stint_df in df.groupby("manager_stint_label", sort=False):
        stint_df = stint_df.sort_values(["from", "elo"]).reset_index(drop=True)
        manager_name = stint_df["manager"].iloc[0]
        high_idx = stint_df["elo"].idxmax()
        low_idx = stint_df["elo"].idxmin()
        high_row = stint_df.loc[high_idx]
        low_row = stint_df.loc[low_idx]

        if high_idx == low_idx:
            extrema_rows.append(
                {
                    "manager": manager_name,
                    "manager_stint_label": stint_label,
                    "from": high_row["from"],
                    "elo": high_row["elo"],
                    "point_type": "High/Low",
                    "label": f"High/Low {round(high_row['elo'])}",
                }
            )
        else:
            extrema_rows.extend(
                [
                    {
                        "manager": manager_name,
                        "manager_stint_label": stint_label,
                        "from": high_row["from"],
                        "elo": high_row["elo"],
                        "point_type": "High",
                        "label": f"High {round(high_row['elo'])}",
                    },
                    {
                        "manager": manager_name,
                        "manager_stint_label": stint_label,
                        "from": low_row["from"],
                        "elo": low_row["elo"],
                        "point_type": "Low",
                        "label": f"Low {round(low_row['elo'])}",
                    },
                ]
            )

    return pd.DataFrame(extrema_rows)

def get_stint_start_finish(df):
    summary_rows = []
    for stint_label, stint_df in df.groupby("manager_stint_label", sort=False):
        stint_df = stint_df.sort_values(["from", "elo"]).reset_index(drop=True)
        manager_name = stint_df["manager"].iloc[0]
        start_row = stint_df.iloc[0]
        finish_row = stint_df.iloc[-1]

        if start_row["from"] == finish_row["from"] and start_row["elo"] == finish_row["elo"]:
            summary_rows.append(
                {
                    "manager": manager_name,
                    "manager_stint_label": stint_label,
                    "from": start_row["from"],
                    "elo": start_row["elo"],
                    "point_type": "Start/Finish",
                    "label": f"Start/Finish {round(start_row['elo'])}",
                }
            )
        else:
            summary_rows.extend(
                [
                    {
                        "manager": manager_name,
                        "manager_stint_label": stint_label,
                        "from": start_row["from"],
                        "elo": start_row["elo"],
                        "point_type": "Start",
                        "label": f"Start {round(start_row['elo'])}",
                    },
                    {
                        "manager": manager_name,
                        "manager_stint_label": stint_label,
                        "from": finish_row["from"],
                        "elo": finish_row["elo"],
                        "point_type": "Finish",
                        "label": f"Finish {round(finish_row['elo'])}",
                    },
                ]
            )

    return pd.DataFrame(summary_rows)

def build_manager_chart(df, title, showlegend=True, include_start_finish=True):
    fig = go.Figure()
    stint_meta = (
        df[["manager_stint_id", "manager", "manager_stint_label"]]
        .drop_duplicates()
        .sort_values("manager_stint_id")
    )
    shown_managers = set()

    for _, meta_row in stint_meta.iterrows():
        stint_id = meta_row["manager_stint_id"]
        manager_name = meta_row["manager"]
        stint_label = meta_row["manager_stint_label"]
        stint_df = df.loc[df["manager_stint_id"] == stint_id].sort_values("from")
        if stint_df.empty:
            continue

        trace_showlegend = showlegend and manager_name not in shown_managers
        shown_managers.add(manager_name)

        fig.add_trace(
            go.Scatter(
                x=stint_df["from"],
                y=stint_df["elo"],
                mode="lines",
                name=manager_name,
                legendgroup=manager_name,
                showlegend=trace_showlegend,
                line=dict(color=manager_colors[manager_name], width=3),
                customdata=np.column_stack([[stint_label] * len(stint_df)]),
                hovertemplate="%{customdata[0]}<br>Date: %{x|%Y-%m-%d}<br>Elo: %{y:.1f}<extra></extra>",
            )
        )

        avg_elo = stint_df["elo"].mean()
        fig.add_trace(
            go.Scatter(
                x=[stint_df["from"].iloc[0], stint_df["from"].iloc[-1]],
                y=[avg_elo, avg_elo],
                mode="lines+text",
                text=["", f"Avg {round(avg_elo)}"],
                textposition="middle right",
                showlegend=False,
                legendgroup=manager_name,
                line=dict(color=manager_colors[manager_name], width=2, dash="dash"),
                opacity=0.7,
                hovertemplate=(
                    stint_label + "<br>Average Elo: " + f"{avg_elo:.1f}" + "<extra></extra>"
                ),
            )
        )

    marker_frames = [get_stint_extrema(df)]
    if include_start_finish:
        marker_frames.append(get_stint_start_finish(df))
    marker_df = pd.concat(marker_frames, ignore_index=True)

    point_styles = {
        "High": ("triangle-up", "top center"),
        "Low": ("triangle-down", "bottom center"),
        "High/Low": ("diamond", "top center"),
        "Start": ("circle", "bottom left"),
        "Finish": ("square", "bottom right"),
        "Start/Finish": ("diamond-open", "bottom center"),
    }

    for point_type, (symbol, textposition) in point_styles.items():
        point_df = marker_df.loc[marker_df["point_type"] == point_type]
        if point_df.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=point_df["from"],
                y=point_df["elo"],
                mode="markers+text",
                text=point_df["label"],
                textposition=textposition,
                textfont=dict(size=10),
                showlegend=False,
                customdata=np.column_stack([point_df["manager_stint_label"], point_df["point_type"]]),
                hovertemplate="%{customdata[0]}<br>%{customdata[1]}<br>Date: %{x|%Y-%m-%d}<br>Elo: %{y:.1f}<extra></extra>",
                marker=dict(
                    size=12,
                    symbol=symbol,
                    color=[manager_colors[manager] for manager in point_df["manager"]],
                    line=dict(color="white", width=1),
                ),
            )
        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        hovermode="x unified",
        xaxis_title="Date",
        yaxis_title="Elo rating",
        legend_title="Manager",
        height=650,
    )
    return fig

combined_fig = build_manager_chart(
    post_fergie,
    "Man United Elo rating over the years post Sir Alex Ferguson",
    showlegend=True,
    include_start_finish=False,
)
combined_fig.show()

for manager in manager_order:
    manager_df = post_fergie.loc[post_fergie["manager"] == manager].copy()
    if manager_df.empty:
        continue

    manager_fig = build_manager_chart(
        manager_df,
        f"{manager}: Man United Elo rating",
        showlegend=False,
        include_start_finish=True,
    )
    manager_fig.show()

<Figure size 640x480 with 0 Axes>

In [53]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

